In [ ]:
# Monitor 01 – inspección robusta de outputs CSV del generador code_01
# Soporta el flujo anterior (full/full_imp/full_diff) y el nuevo MAC
# (full_diff, series_long, selected_curriculum_table, curriculum_stats_all_series).

import re
from urllib.parse import urlparse

import boto3
import pandas as pd
import s3fs

BUCKET_NAME   = "your-private-bucket"
PATH_CSV_BASE = "data/sandboxes/m6hn/data/coe_liquidez_diaria/csv/"

s3 = boto3.client("s3")
fs = s3fs.S3FileSystem(anon=False)

def list_compat_folders():
    paginator = s3.get_paginator("list_objects_v2")
    folders = []
    for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix=PATH_CSV_BASE, Delimiter="/"):
        for cp in page.get("CommonPrefixes", []):
            folder = cp["Prefix"].replace(PATH_CSV_BASE, "").rstrip("/")
            if folder:
                folders.append(folder)
    pattern = re.compile(r"^\d{8}_\d{8}_\d{8}$")
    return sorted(f for f in folders if pattern.match(f))

def exists_s3(path_s3: str) -> bool:
    parsed = urlparse(path_s3)
    bucket, key = parsed.netloc, parsed.path.lstrip("/")
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except Exception:
        try:
            return bool(fs.ls(f"{bucket}/{key}"))
        except Exception:
            return False

def read_csv_s3(path_s3: str, **pd_kw) -> pd.DataFrame:
    parsed = urlparse(path_s3)
    bucket, key = parsed.netloc, parsed.path.lstrip("/")
    try:
        objs = fs.ls(f"{bucket}/{key}")
    except Exception:
        objs = []
    part_files = [o for o in objs if o.endswith(".csv")]
    if part_files:
        frames = []
        for fk in sorted(part_files):
            with fs.open(fk, "rb") as f:
                frames.append(pd.read_csv(f, **pd_kw))
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    with fs.open(f"{bucket}/{key}", "rb") as f:
        return pd.read_csv(f, **pd_kw)

folders = list_compat_folders()
if not folders:
    raise RuntimeError(f"No hay carpetas válidas bajo s3://{BUCKET_NAME}/{PATH_CSV_BASE}")

latest_folder = folders[-1]
base_uri = f"s3://{BUCKET_NAME}/{PATH_CSV_BASE}{latest_folder}/"
print("📂 Carpeta más nueva:", latest_folder)
print("🔗 Base:", base_uri)

candidate_files = [
    "full.csv",
    "full_imp.csv",
    "full_diff.csv",
    "series_long.csv",
    "selected_curriculum_table.csv",
    "curriculum_stats_all_series.csv",
]

dataframes = {}
for name in candidate_files:
    path = base_uri + name
    if exists_s3(path):
        try:
            df = read_csv_s3(path, low_memory=False)
            dataframes[name] = df
            print(f"✔️ {name:35s} shape={df.shape}")
        except Exception as exc:
            print(f"⚠️ {name:35s} error={exc}")
    else:
        print(f"· {name:35s} no existe en esta corrida")

df_full = dataframes.get("full.csv", pd.DataFrame())
df_imp = dataframes.get("full_imp.csv", pd.DataFrame())
df_diff = dataframes.get("full_diff.csv", pd.DataFrame())
df_series_long = dataframes.get("series_long.csv", pd.DataFrame())
df_selected_curriculum = dataframes.get("selected_curriculum_table.csv", pd.DataFrame())
df_curriculum_stats = dataframes.get("curriculum_stats_all_series.csv", pd.DataFrame())

if not df_diff.empty:
    print("\n=== QC full_diff.csv ===")
    print("shape:", df_diff.shape)
    print("fecha min/max:", pd.to_datetime(df_diff["fecha"], errors="coerce").min(), "→", pd.to_datetime(df_diff["fecha"], errors="coerce").max())
    value_cols = [c for c in df_diff.columns if c != "fecha"]
    qc = pd.DataFrame({
        "non_null": df_diff[value_cols].notna().sum(),
        "n_unique": df_diff[value_cols].nunique(dropna=True),
        "null_rate": df_diff[value_cols].isna().mean(),
        "zero_rate": (df_diff[value_cols].fillna(0) == 0).mean(),
    }).sort_values(["non_null", "n_unique"])
    display(qc.head(30))
    display(df_diff.head())
else:
    print("⚠️ No se encontró full_diff.csv; code_02 no tendrá insumo compatible.")


In [ ]:
df_diff

In [ ]:
df_series_long

In [ ]:
df_selected_curriculum

In [ ]:
df_curriculum_stats